# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 16 — A threshold-free measure, and seven checks the manuscript owes

The manuscript states in its limitations that a threshold-free measure of agreement "would
remove both dependencies and is the measure we would choose if starting again", and that
agreement is measured only on mutagenic compounds and without conditioning on whether the
prediction was correct. Declaring a better measure and not computing it is weaker than
computing it, and everything needed is already stored.

This notebook adds the measure and settles six further points, none of which needs training:

1. Environment and stored quantities
2. Agreement as an area under the ROC curve
3. Correctly classified molecules, and a specificity check on non-mutagenic compounds
4. Untrained means resampled over molecules as well as draws
5. The attribution baseline at five hundred draws, with intervals
6. The encoding ablation under the on-manifold baseline
7. How much the regime split depends on how the fused-aromatic pattern is written
8. An empirical intraclass correlation for the design effect
9. Resemblance against advantage, within architectures
10. Summary

*Runtime.* Section 5 builds three thousand networks and runs Integrated Gradients on each,
which is an overnight job on CPU; run this notebook on a GPU, or lower `N_DRAWS_BASELINE`
below to 200, which is still four times the fifty draws the manuscript currently reports. The
other seven stages take one to two hours on CPU. Every stage checkpoints and writes its own
table, so a failure late in the notebook does not cost the earlier results.

## Pre-registered readings

Every stage below has a reading fixed before the numbers were seen. The paper argues that
references should be chosen in advance, and the same discipline applies to its own analyses.
Where a reading depends on a comparison, the criterion is whether the interval on the
difference excludes zero, not whether one point estimate is larger than another.

| stage | if | then |
|---|---|---|
| AUROC, GIN minus degree | interval excludes zero, positive | the aggregate comparison is reported on the AUROC scale and the lift comparison becomes a sensitivity analysis |
| | interval contains zero | both scales agree that the aggregate does not separate, and the paper says so on the threshold-free measure too |
| | interval excludes zero, negative | the trained model is worse than the baseline on the whole ranking, which strengthens the paper's claim and must be reported as such |
| Other explainers, AUROC scale | TreeSHAP interval excludes zero and is positive | the scope sentence in the abstract stands as written |
| | TreeSHAP interval contains zero | the claim narrows to gradient attributions on graphs *at the reference fraction*, and the abstract changes |
| Correctly classified molecules | agreement rises | the misclassified molecules were diluting it; this becomes the cleaner estimate and the unfiltered figure moves to the Supporting Information |
| | agreement unchanged | the filter makes no difference and the unfiltered figure stands |
| | agreement falls | attributions are less alert-aligned where the model is right, which cuts against the reading that agreement reflects learned chemistry; it goes in the results and in the discussion |
| Non-mutagenic compounds, signed | the trained difference from degree is more negative on the non-mutagenic set than on the mutagenic one, and the two intervals do not overlap | evidence that the model uses the chemistry rather than the topology |
| | the two intervals overlap | no evidence either way from this design; reported as such, not as a null result |
| | degree itself differs between the sets | the comparison is confounded by composition and the whole check is reported as uninterpretable |
| Untrained means, both axes | interval still excludes chance | the sentence about the population effect stands |
| | interval includes chance for any architecture | the sentence is narrowed to the architectures where it holds |
| On-manifold baseline at 500 draws | ordering of the reduced encodings still reverses | the caveat in the discussion stands as written |
| | ordering no longer reverses | the caveat is withdrawn and the fifty-draw result is reported as noise |
| Trained encoding, mean baseline | association survives | the abstract sentence loses its qualifier |
| | association collapses | the abstract sentence is removed and the ablation is reported as baseline-dependent |
| Fused-aromatic reading | GIN stays below degree under every reading | the regime result is robust to how the alert is encoded |
| | the sign flips at some reading | the regime result is reported as specific to the broad reading, and the abstract changes |
| Design effect | — | both quantities are reported with what each measures. The intraclass correlation of per-partition estimates and the shared fraction of compounds are not the same thing, so neither agreement nor disagreement between them supports or refutes the approximation. The manuscript will name which one it uses and why |
| Resemblance against advantage | combined within-architecture interval excludes zero | the panel stays with the number quoted |
| | it includes zero | the panel is dropped and the sentence with it |

The two readings that would most change the paper are the first and the fused-aromatic one.
If the AUROC difference contains zero, the central comparison is confirmed on a measure that
does not depend on a threshold. If the fused-aromatic sign flips, the second of the two
headline effects is an artefact of our encoding and the abstract must say so.

*This table is committed before the notebook is run. The manuscript states in its limitations
that its pre-specification claims carry no timestamped record; for these eight analyses, this
is that record.*

## 1. Environment and stored quantities

In [1]:
!pip install -q rdkit
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 42.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
os.makedirs(R, exist_ok=True)
print("results:", R)

Mounted at /content/drive
results: /content/drive/MyDrive/ames_pipeline/results


In [3]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True,
              in_dim=ATOM_DIM):
    return GNN(kind, in_dim=in_dim, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


In [4]:
import sys
sys.path.insert(0, "/content")
from ames9 import *
import sys, json, glob, time
import numpy as np, pandas as pd
from collections import defaultdict
from scipy import stats
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

SEED, PRIMARY_SPLIT_SEED = 42, 101
K_FRAC, K_GRID, N_TIEBREAK, N_BOOT = 0.30, [0.10,0.20,0.30,0.40,0.50], 200, 10000

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, PRIMARY_SPLIT_SEED)]
assert set(df_test["canonical"]) == set(pd.read_csv(f"{R}/primary_test.csv")["canonical"])

TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}
REGIMES = {"full": list(TOX), "no_polycyclic": [k for k in TOX if k != "polycyclic_aromatic"],
           "polycyclic_only": ["polycyclic_aromatic"]}

def build_eval(frame, alerts):
    pats = {k: PAT[k] for k in alerts}; ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab != 1: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

EVAL, CHANCE = build_eval(df_test, REGIMES["full"])
N_EVAL = len(EVAL)
DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]
SMI2I = {e["smiles"]: i for i, e in enumerate(EVAL)}
print(f"evaluation set {N_EVAL}, chance {CHANCE.mean():.4f}")

def top_k(scores, k_frac, tie_seed):
    """Index sets of the highest-ranked k fraction, one per molecule."""
    rng = np.random.default_rng(tie_seed); out = []
    for e, sc in zip(EVAL, scores):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out.append(set(order[:k].tolist()))
    return out

def overlap_vec(scores, k_frac=K_FRAC, tie_seed=0, ev=None, ch=None):
    ev = EVAL if ev is None else ev
    rng = np.random.default_rng(tie_seed); out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def mean_overlap(scores, k_frac=K_FRAC, n=N_TIEBREAK, base=0, ev=None):
    return np.nanmean([overlap_vec(scores, k_frac, base+j, ev) for j in range(n)], axis=0)

def lift_of(ov, ch=None):
    ch = CHANCE if ch is None else ch
    m = ~np.isnan(ov)
    return float(ov[m].mean()/ch[m].mean()) if m.sum() else np.nan

evaluation set 237, chance 0.4360


In [5]:
N_DRAWS_BASELINE = 500      # lower to 200 if running on CPU
CKPT = f"{WORK}/ckpt_part16"
os.makedirs(CKPT, exist_ok=True)

def cached(name, fn):
    """Each stage writes its own table; a rerun skips what is already on disk."""
    p = f"{R}/{name}"
    if os.path.exists(p):
        print(f"  {name}: loaded from disk")
        return pd.read_csv(p)
    out = fn()
    if "note" in out.columns or not len(out):
        print(f"  {name}: stage could not run, not cached")
        return out
    out.to_csv(p, index=False)
    print(f"  {name}: written")
    return out

y_test = df_test["label"].values
print(f"test compounds {len(df_test)}, mutagenic {int(y_test.sum())}, "
      f"non-mutagenic {int((y_test==0).sum())}")

# a three-hour run should not end in "skipped"; check the stored scores before starting
present = sorted(os.path.basename(p) for p in glob.glob(f"{EXPL}/*.npz"))
print(f"\n{len(present)} score files in {EXPL}")
EXPECTED = {"Integrated Gradients": "scores_ig_seed*.npz",
            "TreeSHAP":             "scores_shap_atom.npz",
            "Gradient x Input":     "scores_gradinput_seed*.npz",
            "Saliency":             "scores_saliency_seed*.npz",
            "GNNExplainer, default": "scores_gnnexplainer_default*.npz",
            "GNNExplainer, longer":  "scores_gnnexplainer_longer*.npz",
            "GNNExplainer, weak":    "scores_gnnexplainer_weak*.npz"}
for lab, pat in EXPECTED.items():
    n = len(glob.glob(f"{EXPL}/{pat}"))
    print(f"  {lab:22s} {pat:32s} {n} file(s)" + ("" if n else "   <- missing"))
assert len(glob.glob(f"{EXPL}/scores_ig_seed*.npz")) >= 20, "Part 12 scores missing"

test compounds 726, mutagenic 336, non-mutagenic 390

161 score files in /content/drive/MyDrive/ames_pipeline/explain
  Integrated Gradients   scores_ig_seed*.npz              20 file(s)
  TreeSHAP               scores_shap_atom.npz             0 file(s)   <- missing
  Gradient x Input       scores_gradinput_seed*.npz       20 file(s)
  Saliency               scores_saliency_seed*.npz        20 file(s)
  GNNExplainer, default  scores_gnnexplainer_default*.npz 0 file(s)   <- missing
  GNNExplainer, longer   scores_gnnexplainer_longer*.npz  0 file(s)   <- missing
  GNNExplainer, weak     scores_gnnexplainer_weak*.npz    0 file(s)   <- missing


## 2. Agreement as an area under the ROC curve

Lift at a retained fraction inherits two choices: where to cut the ranking, and how to break
ties within a level. Atom degree takes six distinct values across the dataset and fewer within
a molecule, so a large part of the retained set is filled by tie-breaking rather than by the
ranking itself.

Scoring alert membership by the attribution and taking the area under the ROC curve removes
both. It uses the whole ranking, is invariant to monotone transformations of the score, and
handles ties by the usual mid-rank convention. It is computed per molecule and averaged, so
molecules contribute equally rather than in proportion to their size.

The two measures answer slightly different questions and we report both: lift asks what is in
the top of the ranking, AUROC asks whether alert atoms are ranked above non-alert atoms
anywhere in it.

In [6]:
def grad_scores(model, graphs, method="ig", n_steps=32, batch_size=64, signed=False):
    """Integrated Gradients from a zero baseline.

    The absolute sum over feature dimensions is the convention used throughout this work.
    A signed sum is available because the specificity check needs it: an absolute sum is
    blind to direction, so an atom pushing the prediction towards non-mutagenic scores as
    highly as one pushing towards mutagenic, and the hypothesis that a model ranks alert
    atoms lower in non-mutagenic compounds cannot be tested under it."""
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        total = torch.zeros_like(x0)
        for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
            xa = (x0*al).detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            total += torch.autograd.grad(lg.sum(), xa)[0]
        a_ = x0*(total/n_steps)
        attr = (a_.sum(1) if signed else a_.abs().sum(1)).detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(attr[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

from sklearn.metrics import roc_auc_score as _auc

def per_molecule_auroc(scores, ev=None):
    """Area under the ROC curve for alert membership, one value per molecule."""
    ev = EVAL if ev is None else ev
    out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        y = np.zeros(e["n_atoms"]); y[list(e["tox"])] = 1
        if 0 < y.sum() < len(y):
            out[i] = _auc(y, np.asarray(sc, dtype=float))
    return out

def tie_fraction(scores, k_frac=K_FRAC, ev=None):
    """Share of the retained set that a tie at the cut decides rather than the ranking."""
    ev = EVAL if ev is None else ev
    out = []
    for e, sc in zip(ev, scores):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        v = np.sort(np.asarray(sc, dtype=float))[::-1]
        cut = v[k-1]
        above = int((v > cut).sum()); at = int((v == cut).sum())
        out.append(0.0 if at == 0 else (k - above)/k)
    return np.array(out)

def boot_mean(x, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); x = x[~np.isnan(x)]
    idx = rng.integers(0, len(x), (n_boot, len(x)))
    return x[idx].mean(axis=1)

files = {"gin": [f for f in sorted(glob.glob(f"{EXPL}/scores_ig_seed*.npz"))
                 if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f],
         "gat": sorted(glob.glob(f"{EXPL}/scores_ig_gat_seed*.npz")),
         "gcn": sorted(glob.glob(f"{EXPL}/scores_ig_gcn_seed*.npz"))}
for a_, v_ in files.items():
    assert len(v_) == 20, f"{a_}: {len(v_)} score files, expected 20; rerun Part 12"

def paired_auroc(per, deg_auc, n_boot=N_BOOT, seed=SEED):
    """AUROC(model) - AUROC(degree), molecules and models resampled together."""
    rng = np.random.default_rng(seed)
    n_mod, n_mol = per.shape
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        out[b] = np.nanmean(per[np.ix_(si, mi)]) - np.nanmean(deg_auc[mi])
    return out

def build_auroc():
    """Every arm on one scale, and the difference from the baseline with its interval."""
    rows = []
    d_auc = per_molecule_auroc(DEG)
    v = boot_mean(d_auc)
    rows.append({"arm": "atom degree", "n": 1, "auroc": float(np.nanmean(d_auc)),
                 "ci_low": float(np.percentile(v, 2.5)),
                 "ci_high": float(np.percentile(v, 97.5)),
                 "diff_vs_degree": np.nan, "diff_low": np.nan, "diff_high": np.nan,
                 "mean_tie_fraction": float(np.mean(tie_fraction(DEG)))})
    rng = np.random.default_rng(SEED)
    rnd = [rng.random(e["n_atoms"]) for e in EVAL]
    r_auc = per_molecule_auroc(rnd)
    d = paired_auroc(r_auc[None, :], d_auc, seed=SEED+1)
    v = boot_mean(r_auc, seed=SEED+1)
    rows.append({"arm": "random ranking", "n": 1, "auroc": float(np.nanmean(r_auc)),
                 "ci_low": float(np.percentile(v, 2.5)),
                 "ci_high": float(np.percentile(v, 97.5)),
                 "diff_vs_degree": float(np.nanmean(r_auc)-np.nanmean(d_auc)),
                 "diff_low": float(np.percentile(d, 2.5)),
                 "diff_high": float(np.percentile(d, 97.5)),
                 "mean_tie_fraction": float(np.mean(tie_fraction(rnd)))})

    # gradient methods across the twenty models of each architecture
    for arch in ["gin", "gat", "gcn"]:
        per = np.array([per_molecule_auroc([np.load(f)[f"m{i_}"] for i_ in range(N_EVAL)])
                        for f in files[arch]])
        d = paired_auroc(per, d_auc, seed=SEED+7)
        z = np.load(files[arch][0]); sc0 = [z[f"m{i_}"] for i_ in range(N_EVAL)]
        rows.append({"arm": f"trained {arch.upper()}, IG", "n": len(files[arch]),
                     "auroc": float(np.nanmean(per, axis=1).mean()),
                     "ci_low": np.nan, "ci_high": np.nan,
                     "diff_vs_degree": float(np.nanmean(per)-np.nanmean(d_auc)),
                     "diff_low": float(np.percentile(d, 2.5)),
                     "diff_high": float(np.percentile(d, 97.5)),
                     "mean_tie_fraction": float(np.mean(tie_fraction(sc0)))})

    # the other explainers, from the stored scores of the designated model
    for lab, pat in [("TreeSHAP, fingerprint", f"{EXPL}/scores_shap_atom.npz"),
                     ("Gradient x Input", f"{EXPL}/scores_gradinput_seed*.npz"),
                     ("Saliency", f"{EXPL}/scores_saliency_seed*.npz"),
                     ("GNNExplainer, default", f"{EXPL}/scores_gnnexplainer_default*.npz"),
                     ("GNNExplainer, longer", f"{EXPL}/scores_gnnexplainer_longer*.npz"),
                     ("GNNExplainer, weak reg", f"{EXPL}/scores_gnnexplainer_weak*.npz")]:
        fs = sorted(glob.glob(pat)) if "*" in pat else ([pat] if os.path.exists(pat) else [])
        if not fs:
            print(f"    {lab}: no stored scores, skipped"); continue
        per = []
        for f in fs:
            z = np.load(f)
            keys = [k for k in z.files if k.startswith("m")]
            if len(keys) < N_EVAL: continue
            per.append(per_molecule_auroc([z[f"m{i_}"] for i_ in range(N_EVAL)]))
        if not per: continue
        per = np.array(per)
        d = paired_auroc(per, d_auc, seed=SEED+11)
        rows.append({"arm": lab, "n": len(per),
                     "auroc": float(np.nanmean(per, axis=1).mean()),
                     "ci_low": np.nan, "ci_high": np.nan,
                     "diff_vs_degree": float(np.nanmean(per)-np.nanmean(d_auc)),
                     "diff_low": float(np.percentile(d, 2.5)),
                     "diff_high": float(np.percentile(d, 97.5)),
                     "mean_tie_fraction": np.nan})
    out = pd.DataFrame(rows)
    out["excludes_zero"] = (out.diff_low > 0) | (out.diff_high < 0)
    return out

auroc = cached("table77_auroc.csv", build_auroc)
print()
print(auroc.round(4).to_string(index=False))
print()
print("  0.5 is chance on the AUROC scale. The difference column is the quantity that")
print("  replaces the lift comparison: it carries both axes and its interval is the test.")

  table77_auroc.csv: loaded from disk

             arm  n  auroc  ci_low  ci_high  diff_vs_degree  diff_low  diff_high  mean_tie_fraction  excludes_zero
     atom degree  1 0.5803  0.5634   0.5977             NaN       NaN        NaN             0.8682          False
  random ranking  1 0.4917  0.4708   0.5126         -0.0886   -0.1162    -0.0612             0.1755           True
 trained GIN, IG 20 0.6368     NaN      NaN          0.0564    0.0187     0.0926             0.1930           True
 trained GAT, IG 20 0.5952     NaN      NaN          0.0148   -0.0068     0.0365             0.1871          False
 trained GCN, IG 20 0.5982     NaN      NaN          0.0179   -0.0127     0.0482             0.1976          False
Gradient x Input 20 0.6101     NaN      NaN          0.0297   -0.0021     0.0616                NaN          False
        Saliency 20 0.6436     NaN      NaN          0.0633    0.0261     0.1010                NaN           True

  0.5 is chance on the AUROC scale. The 

## 3. Correctly classified molecules, and a specificity check

Two gaps in the evaluation set. It contains only mutagenic compounds, so nothing tests whether
a model avoids alert atoms in molecules that are not mutagenic; and it includes molecules the
model got wrong, whose attributions have no clear reading.

The first is answered by building the same evaluation set from the non-mutagenic test
compounds that carry a partial alert match. A model that has learned the chemistry should rank
alert atoms lower there, or at least no higher; a model tracking degree should behave the
same way in both sets, since degree does not know the label.

In [7]:
def eval_set(frame, labels, alerts=None):
    """Evaluation molecules of a given label, carrying a partial alert match."""
    pats = PAT if alerts is None else {k: PAT[k] for k in alerts}
    ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab not in labels: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t_ = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t_.update(mt)
        if not t_ or len(t_) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t_, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

NEG, NEG_CH = eval_set(df_test, {0})
NEG_DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in NEG]
NEG_GRAPHS = [smiles_to_data(e["smiles"], 0.0) for e in NEG]
POS_GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
print(f"non-mutagenic evaluation set: {len(NEG)} molecules, chance {NEG_CH.mean():.4f}")

pred = np.load(f"{R}/predictions_primary.npz")
p_gin = pred["test_GIN"] if "test_GIN" in pred else None
assert p_gin is None or len(p_gin) == len(df_test), "prediction vector misaligned with test set"

# the threshold that produced the sensitivity and specificity of the accuracy table
THRESHOLD = 0.5
tp = f"{R}/table01_primary_metrics.csv"
if os.path.exists(tp):
    tm = pd.read_csv(tp)
    if "threshold" in tm.columns and "GIN" in set(tm.get("model", [])):
        THRESHOLD = float(tm[tm.model == "GIN"].threshold.iloc[0])
print(f"decision threshold {THRESHOLD:.3f}")

correct = None
if p_gin is not None:
    smi2p = dict(zip(df_test["canonical"], p_gin))
    correct = {e["smiles"] for e in EVAL
               if e["smiles"] in smi2p and smi2p[e["smiles"]] >= THRESHOLD}
    alt = {t_: sum(1 for e in EVAL if e["smiles"] in smi2p and smi2p[e["smiles"]] >= t_)
           for t_ in (0.4, 0.5, 0.6)}
    print(f"of the {N_EVAL} mutagenic evaluation molecules, {len(correct)} are predicted "
          f"mutagenic; at thresholds 0.4/0.5/0.6 the counts are {list(alt.values())}")

MODEL_FILE = {"gin": "gin_attr", "gat": "gat_attr", "gcn": "gcn_attr"}

def load_models(arch, n):
    """Stored weights, so the signed convention can be recomputed on either evaluation set."""
    hp = json.load(open(f"{R}/hpo_gnn.json"))[arch]
    out = []
    for f in files[arch][:n]:
        sd = int(f.split("seed")[-1].split(".")[0])
        w = f"{MODELS}/{MODEL_FILE[arch]}_seed{sd}.pt"
        if not os.path.exists(w): continue
        mdl = build_gnn(arch, hp["hidden"], hp["n_layers"], hp["dropout"],
                        hp.get("residual", True))
        mdl.load_state_dict(torch.load(w, map_location=DEVICE, weights_only=False)["state_dict"]); mdl.eval()
        out.append(mdl)
    return out

N_SPEC = 10          # models per architecture for the specificity check

def build_conditioning():
    """Three questions, all on the same footing.

    (a) does restricting to correctly classified molecules change the agreement;
    (b) do models rank alert atoms lower in non-mutagenic compounds than in mutagenic ones;
    (c) with intervals, since the pre-registered readings are stated as intervals.

    Question (b) needs a signed aggregation on BOTH sets. An absolute sum is blind to
    direction, so an atom pushing towards non-mutagenic scores as highly as one pushing the
    other way, and the two sets would not be on a common scale."""
    rows = []
    d_auc_pos = per_molecule_auroc(DEG)
    d_auc_neg = per_molecule_auroc(NEG_DEG, ev=NEG)

    # (a) correctly classified, absolute convention as everywhere else
    if correct:
        keep = [i_ for i_, e in enumerate(EVAL) if e["smiles"] in correct]
        sub = [EVAL[i_] for i_ in keep]; ch = CHANCE[keep]; dg = [DEG[i_] for i_ in keep]
        d_sub = per_molecule_auroc(dg, ev=sub)
        rows.append({"set": "mutagenic, correctly classified", "convention": "absolute",
                     "n_molecules": len(sub), "n_models": 1, "arm": "atom degree",
                     "lift": lift_of(mean_overlap(dg, base=21000, ev=sub), ch),
                     "auroc": float(np.nanmean(d_sub)),
                     "diff_vs_degree": np.nan, "diff_low": np.nan, "diff_high": np.nan})
        for arch in ["gin", "gat", "gcn"]:
            per, L = [], []
            for f in files[arch]:
                z = np.load(f); sc = [z[f"m{i_}"] for i_ in keep]
                L.append(lift_of(mean_overlap(sc, n=50, base=21000, ev=sub), ch))
                per.append(per_molecule_auroc(sc, ev=sub))
            per = np.array(per)
            d = paired_auroc(per, d_sub, seed=SEED+13)
            rows.append({"set": "mutagenic, correctly classified", "convention": "absolute",
                         "n_molecules": len(sub), "n_models": len(per),
                         "arm": f"trained {arch.upper()}",
                         "lift": float(np.mean(L)), "auroc": float(np.nanmean(per)),
                         "diff_vs_degree": float(np.nanmean(per)-np.nanmean(d_sub)),
                         "diff_low": float(np.percentile(d, 2.5)),
                         "diff_high": float(np.percentile(d, 97.5))})

    # (b) and (c): both sets under the signed convention, same models, same function
    for lab, ev_, ch_, gr_, dg_, dauc_ in [
            ("mutagenic", EVAL, CHANCE, POS_GRAPHS, DEG, d_auc_pos),
            ("non-mutagenic", NEG, NEG_CH, NEG_GRAPHS, NEG_DEG, d_auc_neg)]:
        rows.append({"set": lab, "convention": "signed", "n_molecules": len(ev_),
                     "n_models": 1, "arm": "atom degree",
                     "lift": lift_of(mean_overlap(dg_, base=22000, ev=ev_), ch_),
                     "auroc": float(np.nanmean(dauc_)),
                     "diff_vs_degree": np.nan, "diff_low": np.nan, "diff_high": np.nan})
        for arch in ["gin", "gat", "gcn"]:
            mdls = load_models(arch, N_SPEC)
            if not mdls:
                print(f"    {lab} / {arch}: no stored weights, skipped"); continue
            per, L = [], []
            for mdl in mdls:
                sc = grad_scores(mdl, gr_, "ig", signed=True)
                L.append(lift_of(mean_overlap(sc, n=50, base=22000, ev=ev_), ch_))
                per.append(per_molecule_auroc(sc, ev=ev_))
                del mdl
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            per = np.array(per)
            d = paired_auroc(per, dauc_, seed=SEED+17)
            rows.append({"set": lab, "convention": "signed", "n_molecules": len(ev_),
                         "n_models": len(per), "arm": f"trained {arch.upper()}",
                         "lift": float(np.mean(L)), "auroc": float(np.nanmean(per)),
                         "diff_vs_degree": float(np.nanmean(per)-np.nanmean(dauc_)),
                         "diff_low": float(np.percentile(d, 2.5)),
                         "diff_high": float(np.percentile(d, 97.5))})
            print(f"    {lab:14s} {arch}: auroc {np.nanmean(per):.4f}  "
                  f"diff {np.nanmean(per)-np.nanmean(dauc_):+.4f} "
                  f"[{np.percentile(d,2.5):+.4f}, {np.percentile(d,97.5):+.4f}]")
    out = pd.DataFrame(rows)
    out["excludes_zero"] = (out.diff_low > 0) | (out.diff_high < 0)
    return out

cond = cached("table78_conditioning.csv", build_conditioning)
print()
print(cond.round(4).to_string(index=False))
print()
print("  The two signed rows are the comparison: a model that uses the chemistry should")
print("  rank alert atoms lower in the non-mutagenic set than in the mutagenic one, and the")
print("  degree ranking, which cannot see the label, should behave alike in both.")

non-mutagenic evaluation set: 168 molecules, chance 0.2914
decision threshold 0.500
of the 237 mutagenic evaluation molecules, 205 are predicted mutagenic; at thresholds 0.4/0.5/0.6 the counts are [216, 205, 192]
    mutagenic      gin: auroc 0.6694  diff +0.0890 [+0.0488, +0.1255]
    mutagenic      gat: auroc 0.5487  diff -0.0317 [-0.0711, +0.0062]
    mutagenic      gcn: auroc 0.6113  diff +0.0310 [+0.0002, +0.0609]
    non-mutagenic  gin: auroc 0.6290  diff +0.0393 [-0.0064, +0.0833]
    non-mutagenic  gat: auroc 0.5344  diff -0.0554 [-0.0944, -0.0176]
    non-mutagenic  gcn: auroc 0.6212  diff +0.0314 [-0.0051, +0.0662]
  table78_conditioning.csv: written

                            set convention  n_molecules  n_models         arm   lift  auroc  diff_vs_degree  diff_low  diff_high  excludes_zero
mutagenic, correctly classified   absolute          205         1 atom degree 1.2032 0.5788             NaN       NaN        NaN          False
mutagenic, correctly classified   absolute

## 4. Untrained means over both axes

The manuscript argues that a comparison against a fixed constant is wrong because it excludes
the molecule axis, and then reports intervals on the untrained means that do exactly that.
Those intervals are $1.96\,\sigma/\sqrt{500}$ over draws with the molecules held fixed. Here
molecules and draws are resampled together.

In [8]:
UN = {}
for arch in ["gin", "gat", "gcn"]:
    p = f"{EXPL}/untrained_overlaps_{arch}.npy"
    if os.path.exists(p): UN[arch] = np.load(p)
print("stored untrained overlap matrices:", {k: v.shape for k, v in UN.items()})

def build_untrained():
    rows = []
    for arch, ov in UN.items():
        rng = np.random.default_rng(SEED)
        n_draw, n_mol = ov.shape
        b_both = np.empty(N_BOOT); b_draw = np.empty(N_BOOT)
        for b in range(N_BOOT):
            di = rng.integers(0, n_draw, n_draw)
            mi = rng.integers(0, n_mol, n_mol)
            o = np.nanmean(ov[np.ix_(di, mi)], axis=0); c = CHANCE[mi]
            m_ = ~np.isnan(o); b_both[b] = o[m_].mean()/c[m_].mean()
            o2 = np.nanmean(ov[di], axis=0); m2 = ~np.isnan(o2)
            b_draw[b] = o2[m2].mean()/CHANCE[m2].mean()
        point = lift_of(np.nanmean(ov, axis=0))
        rows.append({"architecture": arch.upper(), "n_draws": n_draw, "lift": point,
                     "draws_only_low": float(np.percentile(b_draw, 2.5)),
                     "draws_only_high": float(np.percentile(b_draw, 97.5)),
                     "both_axes_low": float(np.percentile(b_both, 2.5)),
                     "both_axes_high": float(np.percentile(b_both, 97.5)),
                     "excludes_chance_both": bool(np.percentile(b_both, 2.5) > 1.0)})
    return pd.DataFrame(rows)

untr = cached("table79_untrained_both_axes.csv", build_untrained)
print()
print(untr.round(4).to_string(index=False))
print()
print("  The last column is the claim the manuscript makes. If it is False for an")
print("  architecture, the sentence about the population effect must be narrowed.")

stored untrained overlap matrices: {'gin': (500, 237), 'gat': (500, 237), 'gcn': (500, 237)}
  table79_untrained_both_axes.csv: written

architecture  n_draws   lift  draws_only_low  draws_only_high  both_axes_low  both_axes_high  excludes_chance_both
         GIN      500 1.1156          1.1105           1.1206         1.0737          1.1573                  True
         GAT      500 1.0859          1.0749           1.0966         1.0617          1.1110                  True
         GCN      500 1.0370          1.0245           1.0492         1.0084          1.0639                  True

  The last column is the claim the manuscript makes. If it is False for an
  architecture, the sentence about the population effect must be narrowed.


## 5–6. The attribution baseline, and the encoding ablation under it

The on-manifold baseline carries a caveat in the discussion and a limitation of its own, and
rests on fifty draws without intervals. It is raised to five hundred here. The encoding
ablation for trained models was never repeated under it, so the sentence in the abstract
about degree being withdrawn from the encoding rests on the baseline the manuscript itself
calls off-manifold. Both are settled with stored weights and no training.

In [9]:
HP = json.load(open(f"{R}/hpo_gnn.json"))["gin"]
ENC = {"full": [], "no_degree": [19], "no_degree_no_h": [19, 22]}

def graphs_for_frame(frame, drop):
    """Graphs for an arbitrary frame under a reduced encoding."""
    keep = [{"mol": Chem.MolFromSmiles(s), "smiles": s} for s in frame["canonical"]]
    return _graphs([k for k in keep if k["mol"] is not None and k["mol"].GetNumAtoms() > 0], drop)

def graphs_for(drop):
    return _graphs(EVAL, drop)

def _graphs(items, drop):
    def feats(a):
        f = ([int(a.GetSymbol()==x) for x in
              ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']]
             + [int(str(a.GetHybridization())==x) for x in
                ['SP','SP2','SP3','SP3D','SP3D2','other']]
             + [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
                a.GetTotalNumHs(), int(a.IsInRing())])
        return [v for i, v in enumerate(f) if i not in set(drop)]
    out = []
    for e in items:
        mol = e["mol"]
        x = torch.tensor([feats(a) for a in mol.GetAtoms()], dtype=torch.float)
        ei, ea = [], []
        for b_ in mol.GetBonds():
            i_, j_ = b_.GetBeginAtomIdx(), b_.GetEndAtomIdx()
            bt = b_.GetBondType()
            bf = [int(bt==Chem.BondType.SINGLE), int(bt==Chem.BondType.DOUBLE),
                  int(bt==Chem.BondType.TRIPLE), int(bt==Chem.BondType.AROMATIC),
                  int(b_.GetIsConjugated()), int(b_.IsInRing())]
            ei += [[i_, j_], [j_, i_]]; ea += [bf, bf]
        edge_index = (torch.tensor(ei, dtype=torch.long).t().contiguous() if ei
                      else torch.zeros((2, 0), dtype=torch.long))
        edge_attr = (torch.tensor(ea, dtype=torch.float) if ea
                     else torch.zeros((0, 6), dtype=torch.float))
        out.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                        y=torch.tensor([1.0]), smiles=e["smiles"]))
    return out

GR = {k: graphs_for(v) for k, v in ENC.items()}
# the manuscript calls this the dataset-average atom vector, so it is the training set,
# not the 237 evaluation molecules
def train_mean(drop):
    """Running mean over the training set; the graphs are not kept."""
    tot, n = None, 0
    for g in graphs_for_frame(df_train, drop):
        tot = g.x.sum(0) if tot is None else tot + g.x.sum(0)
        n += g.x.shape[0]
    return (tot/n).unsqueeze(0)

REF = {k: train_mean(v) for k, v in ENC.items()}
for k in ENC: print(f"  {k}: atom dimension {GR[k][0].x.shape[1]}")

def rho_deg(scores):
    v = [stats.spearmanr(s, d).correlation
         if len(set(d)) > 1 and len(set(np.round(np.asarray(s, float), 12))) > 1 else np.nan
         for s, d in zip(scores, DEG)]
    return float(np.nanmean(v))

def ig(model, graphs, baseline="zero", ref=None, n_steps=32, bs=64):
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=bs, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        base = torch.zeros_like(x0) if baseline == "zero" else ref.to(DEVICE).expand_as(x0)
        total = torch.zeros_like(x0)
        for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
            xa = (base + al*(x0-base)).detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            total += torch.autograd.grad(lg.sum(), xa)[0]
        attr = ((x0-base)*(total/n_steps)).abs().sum(1).detach().cpu().numpy()
        bb = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(attr[bb == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

def build_baseline():
    rows = []
    for enc in ENC:
        dim = GR[enc][0].x.shape[1]
        for bl in ["zero", "mean"]:
            rs = []
            for j in range(N_DRAWS_BASELINE):
                set_seed(31000+j)
                mdl = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"],
                                HP.get("residual", True), in_dim=dim)
                rs.append(rho_deg(ig(mdl, GR[enc], bl, REF[enc])))
                del mdl
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            rs = np.array(rs)
            rows.append({"encoding": enc, "baseline": bl, "n_draws": len(rs),
                         "rho_mean": float(rs.mean()), "rho_sd": float(rs.std(ddof=1)),
                         "ci_low": float(np.percentile(rs, 2.5)),
                         "ci_high": float(np.percentile(rs, 97.5))})
            print(f"    {enc:16s} {bl:5s} rho {rs.mean():+.3f} "
                  f"[{np.percentile(rs,2.5):+.3f}, {np.percentile(rs,97.5):+.3f}]")
    return pd.DataFrame(rows)

base500 = cached("table80_baseline_500.csv", build_baseline)

def build_trained_encoding():
    rows = []
    for enc in ENC:
        pat = (f"{MODELS}/gin_attr_seed*.pt" if enc == "full"
               else f"{MODELS}/gin_{enc}_seed*.pt")
        fs = sorted(glob.glob(pat))
        if not fs:
            print(f"    {enc}: no stored weights, skipped"); continue
        dim = GR[enc][0].x.shape[1]
        for bl in ["zero", "mean"]:
            rs = []
            for f in fs:
                w = torch.load(f, map_location=DEVICE, weights_only=False)
                mdl = build_gnn("gin", HP["hidden"], HP["n_layers"], HP["dropout"],
                                HP.get("residual", True), in_dim=dim)
                mdl.load_state_dict(w["state_dict"]); mdl.eval()
                rs.append(rho_deg(ig(mdl, GR[enc], bl, REF[enc])))
                del mdl
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            rs = np.array(rs)
            rows.append({"encoding": enc, "baseline": bl, "n_models": len(rs),
                         "rho_mean": float(rs.mean()), "rho_sd": float(rs.std(ddof=1))})
            print(f"    {enc:16s} {bl:5s} trained rho {rs.mean():+.3f} "
                  f"(n = {len(rs)})")
    return pd.DataFrame(rows)

trenc = cached("table81_trained_encoding_baseline.csv", build_trained_encoding)
print()
print(base500.round(3).to_string(index=False))
print()
print(trenc.round(3).to_string(index=False))

  full: atom dimension 24
  no_degree: atom dimension 23
  no_degree_no_h: atom dimension 22
    full             zero  rho +0.682 [+0.440, +0.825]
    full             mean  rho +0.673 [+0.529, +0.765]
    no_degree        zero  rho +0.497 [+0.138, +0.696]
    no_degree        mean  rho +0.603 [+0.416, +0.726]
    no_degree_no_h   zero  rho +0.656 [+0.507, +0.772]
    no_degree_no_h   mean  rho +0.576 [+0.402, +0.708]
  table80_baseline_500.csv: written
    full             zero  trained rho +0.429 (n = 20)
    full             mean  trained rho +0.380 (n = 20)
    no_degree        zero  trained rho +0.250 (n = 20)
    no_degree        mean  trained rho +0.321 (n = 20)
    no_degree_no_h   zero  trained rho +0.421 (n = 20)
    no_degree_no_h   mean  trained rho +0.230 (n = 20)
  table81_trained_encoding_baseline.csv: written

      encoding baseline  n_draws  rho_mean  rho_sd  ci_low  ci_high
          full     zero      500     0.682   0.102   0.440    0.825
          full     mean  

## 7. How much the regime split depends on the pattern

The pattern for fused aromatic systems supplies 62\% of the alert atoms and defines the
regime in which a trained GIN falls below chance. It is our own encoding, written as
naphthalene, and it is broader than the toxicophore it stands for. Published SMARTS exist for
these alerts but are not machine-readable from here, so we test the sensitivity directly:
the same analysis under three progressively narrower readings of the pattern.

In [10]:
def fused_aromatic_atoms(mol, min_rings=2):
    """Atoms of any fused aromatic ring system with at least min_rings rings.

    A SMARTS query for a named polycyclic aromatic matches only that topology: an
    anthracene query does not match phenanthrene or benzo[a]pyrene, because their rings
    are fused angularly rather than linearly, and those are the systems that matter for
    mutagenicity. Ring perception captures the concept the alert stands for regardless of
    how the rings are arranged."""
    ri = mol.GetRingInfo()
    arom = [set(r) for r in ri.AtomRings()
            if all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in r)]
    if not arom: return set()
    # union-find over rings that share atoms
    parent = list(range(len(arom)))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a in range(len(arom)):
        for b in range(a+1, len(arom)):
            if arom[a] & arom[b]: parent[find(a)] = find(b)
    groups = {}
    for a in range(len(arom)): groups.setdefault(find(a), []).append(a)
    out = set()
    for members in groups.values():
        if len(members) >= min_rings:
            for a in members: out |= arom[a]
    return out

VARIANTS = {"two or more fused rings": 2, "three or more": 3, "four or more": 4}

def build_pattern_sensitivity():
    """How the regime split moves as the fused-aromatic alert is read more narrowly.

    Two arms per reading. 'fused_only' scores against that reading alone. 'no_fused' takes
    the full library and removes only that reading, so atoms belonging to broader fused
    systems stay in the heteroatom regime; this is what changes across readings and is the
    informative arm."""
    rows = []
    hetero = [k for k in TOX if k != "polycyclic_aromatic"]
    for lab, min_rings in VARIANTS.items():
        fused = {e["smiles"]: fused_aromatic_atoms(e["mol"], min_rings) for e in EVAL}
        n_match = sum(1 for v in fused.values() if v)
        for reg in ["fused_only", "no_fused"]:
            ev = []
            for e in EVAL:
                if reg == "fused_only":
                    t = fused[e["smiles"]]
                else:
                    t = set()
                    for k in hetero:
                        for mt in e["mol"].GetSubstructMatches(PAT[k]): t.update(mt)
                    t -= fused[e["smiles"]]          # the reading under test is removed
                if t and len(t) < e["n_atoms"]:
                    ev.append({**e, "tox": t})
            if len(ev) < 20:
                print(f"    {lab:24s} {reg:11s} only {len(ev)} molecules, skipped"); continue
            ch = np.array([len(x["tox"])/x["n_atoms"] for x in ev])
            dg = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float)
                  for e in ev]
            d_ov = mean_overlap(dg, base=24000, ev=ev); d_l = lift_of(d_ov, ch)
            ovs = np.array([mean_overlap([np.load(f)[f"m{SMI2I[e['smiles']]}"] for e in ev],
                                         n=50, base=24000, ev=ev) for f in files["gin"]])
            rng = np.random.default_rng(SEED)
            bb = np.empty(N_BOOT)
            for b in range(N_BOOT):
                mi = rng.integers(0, len(ev), len(ev)); si = rng.integers(0, len(ovs), len(ovs))
                mo = np.nanmean(ovs[np.ix_(si, mi)], axis=0); c = ch[mi]
                m1 = ~np.isnan(mo); m2 = ~np.isnan(d_ov[mi])
                bb[b] = mo[m1].mean()/c[m1].mean() - d_ov[mi][m2].mean()/c[m2].mean()
            m_l = lift_of(np.nanmean(ovs, axis=0), ch)
            rows.append({"reading": lab, "min_rings": min_rings,
                         "molecules_with_system": n_match, "regime": reg, "n_eval": len(ev),
                         "degree_lift": d_l, "gin_lift": m_l, "difference": m_l - d_l,
                         "ci_low": float(np.percentile(bb, 2.5)),
                         "ci_high": float(np.percentile(bb, 97.5))})
            print(f"    {lab:24s} {reg:11s} n={len(ev):3d} diff {m_l-d_l:+.3f} "
                  f"[{np.percentile(bb,2.5):+.3f}, {np.percentile(bb,97.5):+.3f}]")
    out = pd.DataFrame(rows)
    if len(out): out["excludes_zero"] = (out.ci_low > 0) | (out.ci_high < 0)
    return out

patt = cached("table82_pattern_sensitivity.csv", build_pattern_sensitivity)
print()
print(patt.round(3).to_string(index=False))
print()
print("  Ring perception rather than a named SMARTS, so angular systems such as")
print("  phenanthrene and benzo[a]pyrene are counted; a linear-acene query would miss them,")
print("  and those are the systems that carry the mutagenic mechanism.")

    two or more fused rings  fused_only  n=127 diff -0.297 [-0.391, -0.203]
    two or more fused rings  no_fused    n=174 diff +0.517 [+0.364, +0.683]
    three or more            fused_only  n= 76 diff -0.225 [-0.318, -0.139]
    three or more            no_fused    n=174 diff +0.465 [+0.320, +0.621]
    four or more             fused_only  n= 34 diff -0.192 [-0.277, -0.115]
    four or more             no_fused    n=174 diff +0.448 [+0.307, +0.597]
  table82_pattern_sensitivity.csv: written

                reading  min_rings  molecules_with_system     regime  n_eval  degree_lift  gin_lift  difference  ci_low  ci_high  excludes_zero
two or more fused rings          2                    140 fused_only     127        1.189     0.892      -0.297  -0.391   -0.203           True
two or more fused rings          2                    140   no_fused     174        1.214     1.732       0.517   0.364    0.683           True
          three or more          3                     89 fused_only

## 8. An empirical intraclass correlation for the design effect

The effective number of independent partitions uses $k/(1+(k-1)\rho)$ with $\rho$ taken from
the shared fraction of compounds between test sets. That substitution assumes every compound
contributes equally to the variance and that model-level variability is negligible. The
intraclass correlation of the estimates themselves can be computed directly from the
per-partition ROC-AUC values, and the two can be compared.

In [12]:
def build_icc():
    """Two quantities that the manuscript currently conflates, computed side by side."""
    p = f"{R}/benchmark_runs.csv"
    if not os.path.exists(p):
        print("  benchmark_runs.csv not found; the fifteen-partition results are needed")
        return pd.DataFrame([{"note": "benchmark_runs.csv missing"}])
    b = pd.read_csv(p)
    metric = next((c for c in ["ROC_AUC", "test_roc_auc", "roc_auc"] if c in b.columns), None)
    group = next((c for c in ["partition", "split_seed", "seed"] if c in b.columns), None)
    assert metric and group, f"columns not recognised: {list(b.columns)}"
    if "arm" in b.columns and "main" in set(b["arm"]):
        b = b[b["arm"] == "main"]          # the ablation arms are not part of this comparison
    piv = b.pivot_table(index=group, columns="model", values=metric)
    dropped = piv.columns[piv.isna().any()].tolist()
    if dropped: print(f"  models dropped for missing partitions: {dropped}")
    piv = piv.dropna(axis=1)
    k, n_ = piv.shape
    assert k >= 3 and n_ >= 2, f"need at least three partitions and two models, have {k} x {n_}"
    print(f"  {metric} over {k} partitions x {n_} models")

    # one-way random effects: partitions group, models repeat within a partition
    grand = piv.values.mean()
    ms_between = n_*((piv.mean(axis=1)-grand)**2).sum()/(k-1)
    ms_within = ((piv.values - piv.mean(axis=1).values[:, None])**2).sum()/(k*(n_-1))
    icc = max(0.0, (ms_between-ms_within)/(ms_between+(n_-1)*ms_within))

    pt = pd.read_csv(f"{R}/table41_partition_dependence.csv")
    row = pt[pt.scheme.str.contains("size", case=False, na=False)]
    assert len(row) == 1, f"expected one size-ordered scheme, found {len(row)}: {list(pt.scheme)}"
    J = float(row.mean_jaccard.iloc[0]); rho_j = 2*J/(1+J)
    return pd.DataFrame([{"metric": metric, "k_partitions": k, "n_models": n_,
                          "icc_partition_heterogeneity": icc,
                          "n_eff_from_icc": k/(1+(k-1)*icc),
                          "mean_jaccard": J, "rho_from_jaccard": rho_j,
                          "n_eff_from_jaccard": k/(1+(k-1)*rho_j)}])

icc = cached("table83_design_effect.csv", build_icc)
print()
print(icc.round(4).to_string(index=False))
print()
print("  These two quantities are related but not the same, and the table names them so.")
print("  The first is how much partitions differ from one another relative to how much")
print("  models differ within a partition; it rises with the heterogeneity of the test sets")
print("  whether or not those sets overlap. The design effect wants the correlation between")
print("  estimates of the same quantity on two partitions, which is what the shared fraction")
print("  of compounds approximates. Neither agreement nor disagreement between them settles")
print("  whether the approximation is right; both are reported and named.")

  ROC_AUC over 30 partitions x 5 models
  table83_design_effect.csv: written

 metric  k_partitions  n_models  icc_partition_heterogeneity  n_eff_from_icc  mean_jaccard  rho_from_jaccard  n_eff_from_jaccard
ROC_AUC            30         5                       0.9751          1.0247        0.6335            0.7756              1.2769

  These two quantities are related but not the same, and the table names them so.
  The first is how much partitions differ from one another relative to how much
  models differ within a partition; it rises with the heterogeneity of the test sets
  whether or not those sets overlap. The design effect wants the correlation between
  estimates of the same quantity on two partitions, which is what the shared fraction
  of compounds approximates. Neither agreement nor disagreement between them settles
  whether the approximation is right; both are reported and named.


## 9. Resemblance against advantage, within architectures

Panel (b) of the architecture figure plots agreement against the within-molecule resemblance
to a degree ranking, and the text draws a claim from it. Sixty points across three
architectures invite a between-architecture reading that the text elsewhere disowns. The
within-architecture correlation is the quantity that panel would have to support.

In [13]:
def build_resemblance():
    m59 = pd.read_csv(f"{R}/table59_architecture_models.csv")
    rows = []
    for arch in ["gin", "gat", "gcn"]:
        s_ = m59[m59.arch == arch].dropna(subset=["spearman_ig_degree", "lift_ig_k30"])
        r, p = stats.pearsonr(s_.spearman_ig_degree, s_.lift_ig_k30)
        z = np.arctanh(r); se = 1/np.sqrt(len(s_)-3)
        rows.append({"scope": arch.upper(), "n": len(s_), "pearson_r": r,
                     "ci_low": np.tanh(z-1.96*se), "ci_high": np.tanh(z+1.96*se), "p": p})
    s_ = m59.dropna(subset=["spearman_ig_degree", "lift_ig_k30"])
    r, p = stats.pearsonr(s_.spearman_ig_degree, s_.lift_ig_k30)
    z = np.arctanh(r); se = 1/np.sqrt(len(s_)-3)
    rows.append({"scope": "pooled", "n": len(s_), "pearson_r": r,
                 "ci_low": np.tanh(z-1.96*se), "ci_high": np.tanh(z+1.96*se), "p": p})
    w = [x for x in rows if x["scope"] != "pooled"]
    zs = [np.arctanh(x["pearson_r"]) for x in w]; ws = [x["n"]-3 for x in w]
    zc = sum(a*b for a, b in zip(zs, ws))/sum(ws); se = 1/np.sqrt(sum(ws))
    rows.append({"scope": "combined within-architecture", "n": sum(x["n"] for x in w),
                 "pearson_r": np.tanh(zc), "ci_low": np.tanh(zc-1.96*se),
                 "ci_high": np.tanh(zc+1.96*se),
                 "p": 2*(1-stats.norm.cdf(abs(zc)/se))})
    return pd.DataFrame(rows)

resem = cached("table84_resemblance.csv", build_resemblance)
print()
print(resem.round(3).to_string(index=False))

  table84_resemblance.csv: written

                       scope  n  pearson_r  ci_low  ci_high     p
                         GIN 20      0.108  -0.351    0.526 0.649
                         GAT 20      0.226  -0.241    0.608 0.338
                         GCN 20      0.094  -0.363    0.515 0.693
                      pooled 60     -0.005  -0.258    0.249 0.971
combined within-architecture 60      0.143  -0.129    0.396 0.302


## 10. Summary

In [14]:
print("agreement as AUROC")
print(auroc[["arm","n","auroc","diff_vs_degree","diff_low","diff_high",
             "excludes_zero","mean_tie_fraction"]].round(4).to_string(index=False))
print("\nconditioning on label and on correctness")
print(cond[["set","convention","arm","n_molecules","n_models","auroc",
            "diff_vs_degree","diff_low","diff_high","excludes_zero"]]
      .round(4).to_string(index=False))
print("\nuntrained means, both axes")
print(untr[["architecture","lift","both_axes_low","both_axes_high",
            "excludes_chance_both"]].round(4).to_string(index=False))
print("\nattribution baseline at 500 draws")
print(base500.pivot_table(index="encoding", columns="baseline",
                          values=["rho_mean","ci_low","ci_high"]).round(3).to_string())
print("\ntrained encoding under both baselines")
if "encoding" in trenc.columns:
    print(trenc.pivot_table(index="encoding", columns="baseline",
                            values="rho_mean").round(3).to_string())
print("\npattern sensitivity")
if "reading" in patt.columns:
    print(patt[["reading","min_rings","regime","n_eval","difference","ci_low","ci_high"]]
          .round(3).to_string(index=False))
print("\ndesign effect")
print(icc.round(4).to_string(index=False))
print("\nresemblance against advantage")
print(resem[["scope","n","pearson_r","ci_low","ci_high"]].round(3).to_string(index=False))
print("\nWritten to", R)
for f in ["table77_auroc.csv","table78_conditioning.csv","table79_untrained_both_axes.csv",
          "table80_baseline_500.csv","table81_trained_encoding_baseline.csv",
          "table82_pattern_sensitivity.csv","table83_design_effect.csv",
          "table84_resemblance.csv"]:
    print("  -", f, "yes" if os.path.exists(f"{R}/{f}") else "MISSING")
# every quantity that will enter the text belongs in the ledger the manuscript maintains
def ledger_rows():
    out = {}
    for _, r in auroc.iterrows():
        k = r["arm"].lower().replace(" ", "_").replace(",", "").replace("x", "x")
        out[f"auroc_{k}"] = float(r["auroc"])
        if pd.notna(r.get("diff_vs_degree")):
            out[f"aurocdiff_{k}"] = float(r["diff_vs_degree"])
            out[f"aurocdiff_{k}_lo"] = float(r["diff_low"])
            out[f"aurocdiff_{k}_hi"] = float(r["diff_high"])
        if pd.notna(r.get("mean_tie_fraction")):
            out[f"tiefrac_{k}"] = float(r["mean_tie_fraction"])
    for _, r in cond.iterrows():
        k = "_".join([r["set"].split(",")[0], r["convention"], r["arm"]]
                     ).lower().replace(" ", "_").replace(",", "")
        out[f"cond_{k}_lift"] = float(r["lift"]); out[f"cond_{k}_auroc"] = float(r["auroc"])
        if pd.notna(r.get("diff_vs_degree")):
            out[f"cond_{k}_diff"] = float(r["diff_vs_degree"])
            out[f"cond_{k}_lo"] = float(r["diff_low"])
            out[f"cond_{k}_hi"] = float(r["diff_high"])
    for _, r in untr.iterrows():
        a = r["architecture"].lower()
        out[f"untrboth_{a}_lo"] = float(r["both_axes_low"])
        out[f"untrboth_{a}_hi"] = float(r["both_axes_high"])
    for _, r in base500.iterrows():
        out[f"base500_{r['encoding']}_{r['baseline']}"] = float(r["rho_mean"])
        out[f"base500_{r['encoding']}_{r['baseline']}_lo"] = float(r["ci_low"])
    if "encoding" in trenc.columns:
        for _, r in trenc.iterrows():
            out[f"trenc_{r['encoding']}_{r['baseline']}"] = float(r["rho_mean"])
    if "reading" in patt.columns:
        for _, r in patt.iterrows():
            k = f"{r['min_rings']}_{r['regime']}"
            out[f"patt_{k}_diff"] = float(r["difference"])
            out[f"patt_{k}_lo"] = float(r["ci_low"]); out[f"patt_{k}_hi"] = float(r["ci_high"])
    if "icc_partition_heterogeneity" in icc.columns:
        out["icc_partition_heterogeneity"] = float(icc.iloc[0]["icc_partition_heterogeneity"])
        out["n_eff_from_icc"] = float(icc.iloc[0]["n_eff_from_icc"])
    for _, r in resem.iterrows():
        k = r["scope"].lower().replace(" ", "_").replace("-", "_")
        out[f"resem_{k}_r"] = float(r["pearson_r"])
        out[f"resem_{k}_lo"] = float(r["ci_low"]); out[f"resem_{k}_hi"] = float(r["ci_high"])
    return out

new = ledger_rows()
pd.DataFrame(sorted(new.items()), columns=["quantity", "value"]).to_csv(
    f"{R}/table85_part16_ledger.csv", index=False)
print(f"{len(new)} quantities written to table85_part16_ledger.csv for merging into the")
print("manuscript ledger; nothing from this notebook should reach the text without it.")

import shutil
from google.colab import files
shutil.make_archive("/content/part16_results", "zip", R)
files.download("/content/part16_results.zip")

agreement as AUROC
             arm  n  auroc  diff_vs_degree  diff_low  diff_high  excludes_zero  mean_tie_fraction
     atom degree  1 0.5803             NaN       NaN        NaN          False             0.8682
  random ranking  1 0.4917         -0.0886   -0.1162    -0.0612           True             0.1755
 trained GIN, IG 20 0.6368          0.0564    0.0187     0.0926           True             0.1930
 trained GAT, IG 20 0.5952          0.0148   -0.0068     0.0365          False             0.1871
 trained GCN, IG 20 0.5982          0.0179   -0.0127     0.0482          False             0.1976
Gradient x Input 20 0.6101          0.0297   -0.0021     0.0616          False                NaN
        Saliency 20 0.6436          0.0633    0.0261     0.1010           True                NaN

conditioning on label and on correctness
                            set convention         arm  n_molecules  n_models  auroc  diff_vs_degree  diff_low  diff_high  excludes_zero
mutagenic, correct

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>